# FootShellGaussian - Cross-Section Pseudo-Last Debug Playground

This notebook mirrors the GShell playground, but points at the current pseudo-last cross-section experiment. It is meant for interactive debugging, not for launching the long tmux training job.

## 0 - Environment and cache setup

In [1]:
import os
from pathlib import Path

cache_root = "/data/abelde"
project_root = Path("/data/abelde/projects/active/Shell_Gaussian")
footshell_root = project_root / "FootShellGaussian"
gshell_env = project_root / "baselines" / "GShell" / "GShell_env"

# Set this before running the cell if you want a different GPU:
# NOTEBOOK_CUDA_VISIBLE_DEVICES = "7"
os.environ["CUDA_VISIBLE_DEVICES"] = globals().get("NOTEBOOK_CUDA_VISIBLE_DEVICES", "1")

os.environ["PIP_CACHE_DIR"] = os.path.join(cache_root, ".cache", "pip")
os.environ["TORCH_HOME"] = os.path.join(cache_root, ".cache", "torch")
os.environ["HF_HOME"] = os.path.join(cache_root, ".cache", "huggingface")
os.environ["XDG_CACHE_HOME"] = os.path.join(cache_root, ".cache")
os.environ["TMPDIR"] = os.path.join(cache_root, "tmp")
os.environ["CONDA_PKGS_DIRS"] = os.path.join(cache_root, ".conda", "pkgs")

for d in [
    os.environ["PIP_CACHE_DIR"], os.environ["TORCH_HOME"], os.environ["HF_HOME"],
    os.environ["XDG_CACHE_HOME"], os.environ["TMPDIR"], os.environ["CONDA_PKGS_DIRS"],
]:
    os.makedirs(d, exist_ok=True)

conda_gcc = gshell_env / "bin" / "x86_64-conda-linux-gnu-gcc"
conda_gxx = gshell_env / "bin" / "x86_64-conda-linux-gnu-g++"
os.environ["PATH"] = f"{gshell_env / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["CC"] = str(conda_gcc)
os.environ["CXX"] = str(conda_gxx)
os.environ["CUDAHOSTCXX"] = str(conda_gxx)

lib_prefixes = [
    "/usr/lib/x86_64-linux-gnu",
    str(gshell_env / "lib"),
    str(gshell_env / "lib64"),
]
for key in ["LIBRARY_PATH", "LD_LIBRARY_PATH"]:
    existing = os.environ.get(key, "")
    os.environ[key] = ":".join(lib_prefixes + ([existing] if existing else []))

print("Cache root:", cache_root)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("GShell env:", gshell_env)
print("CC exists:", Path(os.environ["CC"]).exists(), os.environ["CC"])
print("CXX exists:", Path(os.environ["CXX"]).exists(), os.environ["CXX"])

Cache root: /data/abelde
CUDA_VISIBLE_DEVICES: 1
GShell env: /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env
CC exists: True /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/x86_64-conda-linux-gnu-gcc
CXX exists: True /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/x86_64-conda-linux-gnu-g++


## 1 - Imports and paths

In [2]:
import os, sys, time, json, argparse, re
from pathlib import Path
import numpy as np

project_root = "/data/abelde/projects/active/Shell_Gaussian"
repo_root = os.path.join(project_root, "FootShellGaussian")

# Keep FootShellGaussian ahead of baselines/GShell on sys.path. This matters because
# both repos have packages named geometry/render/dataset.
sys.path = [p for p in sys.path if p != repo_root]
sys.path.insert(0, repo_root)
os.chdir(repo_root)

print("Python executable:", sys.executable)
print("Working directory:", os.getcwd())
print("Warming up torch...")
t0 = time.time()
import torch
print(f"torch imported in {time.time() - t0:.1f}s | version: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Logical cuda:0 maps to physical GPU:", os.environ.get("CUDA_VISIBLE_DEVICES", "all visible GPUs"))

import nvdiffrast.torch as dr
import xatlas

from dataset.dataset_nerf_colmap import DatasetNERF
from geometry.gshell_tets_geometry import GShellTetsGeometry
from render import renderutils as ru
from render import obj, material, mesh, texture, mlptexture, light, render, util
from denoiser.denoiser import BilateralDenoiser
from train_gshelltet_polycam import createLoss, prepare_batch, initial_guess_material, validate_itr

print("All imports OK")

Python executable: /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/python
Working directory: /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian
Warming up torch...
torch imported in 0.9s | version: 1.13.1 | CUDA: True
GPU: NVIDIA A100-SXM4-80GB
Logical cuda:0 maps to physical GPU: 1


Detected CUDA files, patching ldflags
Emitting ninja build file /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/render/optixutils/build/build.ninja...
Building extension module optixutils_plugin...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/4] /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/nvcc  -ccbin /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/x86_64-conda-linux-gnu-gcc -DTORCH_EXTENSION_NAME=optixutils_plugin -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -I/data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/render/optixutils/include -isystem /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/lib/python3.10/site-packages/torch/include -isystem /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/lib/python3.10/site-packages/torch/include/TH -isystem /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/lib/python3.10/site-packages/torch/include/THC -isystem 

Loading extension module optixutils_plugin...


All imports OK


## 2 - Configuration

In [7]:
# Main knobs for this notebook.
SHOE_NAME = "Air-Jordan-1-Mid-Wear-Away-Chicago-Gs"
CONFIG_NAME = "shoes_mc_pseudolast_xsec_gt_512.json"

config_path = os.path.join(repo_root, "configs", CONFIG_NAME)
data_path = f"/data/abelde/datasets/processed/gshell_shoes_size_metadata/{SHOE_NAME}"
output_dir = os.path.join(repo_root, "output", "debug_playground", f"{SHOE_NAME}")

pseudo_last_root = os.path.join(repo_root, "output/pseudo_last_section_loft", SHOE_NAME)
pseudo_last_sdf_path = os.path.join(pseudo_last_root, "pseudo_last_sdf.npz")
pseudo_last_sections_path = os.path.join(pseudo_last_root, "pseudo_last_sections.npz")
training_log_path = os.path.join(repo_root, "output", f"{SHOE_NAME}_pseudolast_xsec_gt_512", "logs", "train.log")

os.makedirs(output_dir, exist_ok=True)

# Defaults copied from train_gshelltet_polycam.py, then overwritten by the JSON config.
# This avoids the stale hand-written parser block from older notebook versions.
def make_default_flags():
    defaults = dict(
        config=config_path,
        iter=5000,
        batch=1,
        spp=1,
        layers=1,
        train_res=[512, 512],
        display_res=None,
        texture_res=[1024, 1024],
        display_interval=0,
        save_interval=1000,
        learning_rate=0.01,
        min_roughness=0.08,
        custom_mip=False,
        random_textures=False,
        background="checker",
        loss="logl1",
        out_dir=output_dir,
        ref_mesh=None,
        base_mesh=None,
        validate=True,
        n_samples=4,
        bsdf="pbr",
        denoiser="bilateral",
        denoiser_demodulate=True,
        msdf_reg_open_scale=1e-6,
        msdf_reg_close_scale=3e-4,
        eikonal_scale=5e-3,
        sdf_regularizer=0.2,
        use_foot_prior=False,
        foot_prior_sdf_path="",
        foot_prior_alignment_path="",
        foot_prior_clearance=0.005,
        foot_prior_clearance_weight=2.0,
        foot_prior_msdf_close_weight=0.001,
        foot_prior_msdf_close_margin=0.001,
        foot_prior_plantar_sdf_band=0.035,
        foot_prior_start_iter=500,
        foot_prior_warmup_iter=1000,
        foot_prior_max_surface_points=50000,
        foot_prior_max_watertight_points=50000,
        use_pseudo_last_prior=False,
        pseudo_last_sdf_path="",
        pseudo_last_sections_path="",
        pseudo_last_prior_mode="cross_section",
        pseudo_last_xsec_start_iter=0,
        pseudo_last_xsec_warmup_iter=250,
        pseudo_last_xsec_material_weight=20.0,
        pseudo_last_xsec_empty_weight=10.0,
        pseudo_last_xsec_surface_weight=2.0,
        pseudo_last_xsec_msdf_keep_weight=0.01,
        pseudo_last_xsec_x_slices=64,
        pseudo_last_xsec_y_samples=96,
        pseudo_last_xsec_z_samples=96,
        pseudo_last_xsec_max_points=49152,
        pseudo_last_xsec_grid_max_points=0,
        pseudo_last_xsec_material_margin=0.005,
        pseudo_last_xsec_empty_margin=0.005,
        pseudo_last_xsec_surface_band=0.003,
        pseudo_last_xsec_sole_depth=0.025,
        pseudo_last_xsec_plantar_h_ratio=0.12,
        pseudo_last_xsec_lower_wall_h_ratio=0.45,
        pseudo_last_xsec_ignore_high_h_ratio=0.85,
        pseudo_last_xsec_support_lateral_pad=0.05,
        pseudo_last_xsec_msdf_margin=0.005,
        pseudo_last_xsec_last_surface_points=8192,
        pseudo_last_collision_weight=0.25,
        pseudo_last_msdf_keep_weight=0.001,
        pseudo_last_grid_msdf_keep_weight=0.0,
        pseudo_last_containment_weight=0.05,
        pseudo_last_use_field_conditioning=False,
        pseudo_last_msdf_bias_strength=0.0,
        pseudo_last_msdf_bias_distance=0.055,
        pseudo_last_msdf_bias_inside_tolerance=0.006,
        pseudo_last_msdf_bias_dilate_steps=2,
        pseudo_last_msdf_bias_dilate_decay=0.6,
        pseudo_last_sdf_material_weight=0.0,
        pseudo_last_collision_clearance=0.002,
        pseudo_last_msdf_keep_margin=0.001,
        pseudo_last_grid_msdf_keep_margin=0.005,
        pseudo_last_containment_margin=0.001,
        pseudo_last_sdf_material_margin=0.005,
        pseudo_last_band_distance=0.035,
        pseudo_last_inside_tolerance=0.003,
        pseudo_last_plantar_h_ratio=0.10,
        pseudo_last_side_h_ratio=0.35,
        pseudo_last_side_lateral_min=0.70,
        pseudo_last_heel_s_max=0.25,
        pseudo_last_forefoot_s_min=0.60,
        pseudo_last_forefoot_s_max=0.92,
        pseudo_last_start_iter=500,
        pseudo_last_warmup_iter=1000,
        pseudo_last_collision_start_iter=-1,
        pseudo_last_collision_warmup_iter=-1,
        pseudo_last_msdf_keep_start_iter=-1,
        pseudo_last_msdf_keep_warmup_iter=-1,
        pseudo_last_grid_msdf_keep_start_iter=-1,
        pseudo_last_grid_msdf_keep_warmup_iter=-1,
        pseudo_last_containment_start_iter=-1,
        pseudo_last_containment_warmup_iter=-1,
        pseudo_last_sdf_material_start_iter=0,
        pseudo_last_sdf_material_warmup_iter=0,
        pseudo_last_max_surface_points=50000,
        pseudo_last_max_watertight_points=50000,
        pseudo_last_max_last_surface_points=20000,
        pseudo_last_grid_max_points=0,
        trainset_path=data_path,
        testset_path="",
        mtl_override=None,
        gshell_grid=64,
        mesh_scale=3.6,
        envlight=None,
        env_scale=1.0,
        probe_res=256,
        learn_lighting=True,
        display=None,
        transparency=False,
        lock_light=False,
        lock_pos=False,
        laplace="relative",
        laplace_scale=3000.0,
        pre_load=True,
        no_perturbed_nrm=False,
        decorrelated=False,
        kd_min=[0.0, 0.0, 0.0, 0.0],
        kd_max=[1.0, 1.0, 1.0, 1.0],
        ks_min=[0.0, 0.001, 0.0],
        ks_max=[0.0, 1.0, 1.0],
        nrm_min=[-1.0, -1.0, 0.0],
        nrm_max=[1.0, 1.0, 1.0],
        clip_max_norm=0.0,
        cam_near_far=[0.1, 1000.0],
        lambda_kd=0.1,
        lambda_ks=0.05,
        lambda_nrm=0.025,
        lambda_nrm2=0.25,
        lambda_chroma=0.0,
        lambda_diffuse=0.15,
        lambda_specular=0.0025,
        random_lgt=False,
        normal_only=False,
        use_img_2nd_layer=False,
        use_depth=False,
        use_depth_2nd_layer=False,
        use_tanh_deform=False,
        use_sdf_mlp=True,
        use_msdf_mlp=False,
        use_eikonal=True,
        sdf_mlp_pretrain_steps=4000,
        use_mesh_msdf_reg=True,
        sphere_init=False,
        sphere_init_norm=0.5,
        pretrained_sdf_mlp_path="./data/pretrained_mlp_64_polycam.pt",
        n_hidden=6,
        d_hidden=256,
        n_freq=6,
        skip_in=[3],
        use_float16=False,
        visualize_watertight=False,
        local_rank=0,
        multi_gpu=False,
        boxscale=[1, 1, 1],
        aabb=[-1, -1, -1, 1, 1, 1],
    )
    return argparse.Namespace(**defaults)

FLAGS = make_default_flags()
with open(config_path, "r") as f:
    config_data = json.load(f)
for key, val in config_data.items():
    setattr(FLAGS, key, val)

FLAGS.config = config_path
FLAGS.trainset_path = data_path
FLAGS.out_dir = output_dir
FLAGS.pseudo_last_sdf_path = pseudo_last_sdf_path
FLAGS.pseudo_last_sections_path = pseudo_last_sections_path
if FLAGS.display_res is None:
    FLAGS.display_res = FLAGS.train_res
FLAGS.pretrained_sdf_mlp_path = f"./data/pretrained_mlp_{FLAGS.gshell_grid}_polycam.pt"

# Optional notebook-only override. Leave unset to match the training script.
if "NOTEBOOK_SDF_PRETRAIN_STEPS" in globals():
    FLAGS.sdf_mlp_pretrain_steps = int(NOTEBOOK_SDF_PRETRAIN_STEPS)

important = [
    "config", "trainset_path", "out_dir", "train_res", "display_res", "texture_res",
    "gshell_grid", "mesh_scale", "learning_rate", "use_sdf_mlp", "use_msdf_mlp",
    "sdf_mlp_pretrain_steps", "use_pseudo_last_prior", "pseudo_last_prior_mode",
    "pseudo_last_xsec_material_weight", "pseudo_last_xsec_empty_weight",
    "pseudo_last_xsec_surface_weight", "pseudo_last_xsec_msdf_keep_weight",
    "pseudo_last_sdf_path", "pseudo_last_sections_path",
]
print("Config / key FLAGS:")
for key in important:
    print(f"  {key}: {getattr(FLAGS, key)}")
print("pseudo_last_sdf_exists:", os.path.exists(FLAGS.pseudo_last_sdf_path))
print("pseudo_last_sections_exists:", os.path.exists(FLAGS.pseudo_last_sections_path))
print("training_log_path:", training_log_path)

Config / key FLAGS:
  config: /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/configs/shoes_mc_pseudolast_xsec_gt_512.json
  trainset_path: /data/abelde/datasets/processed/gshell_shoes_size_metadata/Air-Jordan-1-Mid-Wear-Away-Chicago-Gs
  out_dir: /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/output/debug_playground/Air-Jordan-1-Mid-Wear-Away-Chicago-Gs
  train_res: [512, 768]
  display_res: [512, 768]
  texture_res: [1024, 1024]
  gshell_grid: 128
  mesh_scale: 1.0
  learning_rate: [0.0025, 0.005]
  use_sdf_mlp: True
  use_msdf_mlp: False
  sdf_mlp_pretrain_steps: 4000
  use_pseudo_last_prior: True
  pseudo_last_prior_mode: cross_section
  pseudo_last_xsec_material_weight: 20.0
  pseudo_last_xsec_empty_weight: 10.0
  pseudo_last_xsec_surface_weight: 2.0
  pseudo_last_xsec_msdf_keep_weight: 0.01
  pseudo_last_sdf_path: /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/output/pseudo_last_section_loft/Air-Jordan-1-Mid-Wear-Away-Chicago-Gs/pseudo_

## 3 - Dataset

In [8]:
data_root = FLAGS.trainset_path

dataset_train = DatasetNERF(os.path.join(data_root, "transforms.json"), FLAGS, examples=int(1e6))
dataset_validate = DatasetNERF(os.path.join(data_root, "transforms.json"), FLAGS)

print(f"Train examples : {len(dataset_train)}")
print(f"Val examples   : {len(dataset_validate)}")

sample = dataset_train[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:12s} {str(tuple(v.shape)):20s} {v.dtype}")
    else:
        print(f"  {k:12s} {type(v).__name__}: {v}")

DatasetNERF: 36 images with shape [1800, 2400]
DatasetNERF: 36 images with shape [1800, 2400]
Train examples : 1000000
Val examples   : 36
  mv           (1, 4, 4)            torch.float32
  mvp          (1, 4, 4)            torch.float32
  campos       (1, 3)               torch.float32
  resolution   list: [512, 768]
  spp          int: 1
  img          (1, 512, 768, 4)     torch.float32


## 4 - Rasterization context, lighting, denoiser

In [9]:
glctx = dr.RasterizeGLContext()

if FLAGS.learn_lighting:
    lgt = light.create_trainable_env_rnd(FLAGS.probe_res, scale=0.0, bias=0.5)
else:
    lgt = light.load_env(FLAGS.envlight, scale=FLAGS.env_scale, res=[FLAGS.probe_res, FLAGS.probe_res])

denoiser = None
if FLAGS.denoiser == "bilateral":
    denoiser = BilateralDenoiser().cuda()
else:
    assert FLAGS.denoiser == "none", f"Invalid denoiser {FLAGS.denoiser}"

print("Light type:", type(lgt).__name__ if lgt is not None else None)
print("Denoiser  :", type(denoiser).__name__ if denoiser else None)

Light type: EnvironmentLight
Denoiser  : BilateralDenoiser


/tmp/ipykernel_1249648/2015170201.py:1: DeprecationWarning: RasterizeGLContext has been deprecated and uses RasterizeCudaContext internally
  glctx = dr.RasterizeGLContext()


## 5 - Geometry and material initialization

In [10]:
geometry = GShellTetsGeometry(FLAGS.gshell_grid, FLAGS.mesh_scale, FLAGS)

print(f"Tet grid res : {FLAGS.gshell_grid}")
print(f"Mesh scale   : {FLAGS.mesh_scale}")
print(f"Verts        : {tuple(geometry.verts.shape)}")
print(f"Indices      : {tuple(geometry.indices.shape)}")
print("Parameters:")
for name, param in geometry.named_parameters():
    print(f"  {name:45s} {str(list(param.shape)):22s} requires_grad={param.requires_grad}")

mat = initial_guess_material(geometry, True, FLAGS, None)
mat["no_perturbed_nrm"] = True
print("Material keys:", list(mat.keys()))

Cuda path /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env
End of OptiXStateWrapper 
using resolution 128
Loaded pseudo-last topology prior:
  SDF: /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/output/pseudo_last_section_loft/Air-Jordan-1-Mid-Wear-Away-Chicago-Gs/pseudo_last_sdf.npz
  sections: /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/output/pseudo_last_section_loft/Air-Jordan-1-Mid-Wear-Away-Chicago-Gs/pseudo_last_sections.npz
  cross-section pool: material=318446, sole=288956, plantar=2325, sidewall=29490, empty=154656, surface=17152
  cross-section grid mSDF candidates: 229 (sole=190, plantar=2, sidewall=39)
  cross-section grid candidate h: p50=-0.00987, p99=0.03966, far_below=0


100%|██████████| 4000/4000 [03:43<00:00, 17.88it/s]


sdf net trained with loss: tensor(2.8620e-07, device='cuda:0', grad_fn=<MeanBackward0>)
Tet grid res : 128
Mesh scale   : 1.0
Verts        : (277410, 3)
Indices      : (1524684, 4)
Parameters:
  sdf                                           [277410]               requires_grad=True
  msdf                                          [277410]               requires_grad=True
  deform                                        [277410, 3]            requires_grad=True
  sdf_net.net.0.weight                          [256, 39]              requires_grad=True
  sdf_net.net.0.bias                            [256]                  requires_grad=True
  sdf_net.net.2.weight                          [256, 256]             requires_grad=True
  sdf_net.net.2.bias                            [256]                  requires_grad=True
  sdf_net.net.4.weight                          [256, 256]             requires_grad=True
  sdf_net.net.4.bias                            [256]                  requires_grad=Tr

## 5a - Cross-section pseudo-last prior inspection

In [ ]:
pseudo_prior = getattr(geometry, "pseudo_last_prior_loss", None)
print("pseudo_prior:", type(pseudo_prior).__name__ if pseudo_prior is not None else None)

if pseudo_prior is not None:
    print("SDF path      :", FLAGS.pseudo_last_sdf_path)
    print("sections path :", FLAGS.pseudo_last_sections_path)
    print("section_x     :", tuple(pseudo_prior.section_x.shape), float(pseudo_prior.section_x[0]), float(pseudo_prior.section_x[-1]))
    print("bottom grid   :", tuple(pseudo_prior.bottom_y_grid.shape))
    print("height        :", tuple(pseudo_prior.section_height.shape), float(pseudo_prior.section_height.min()), float(pseudo_prior.section_height.max()))
    print("last surface  :", tuple(pseudo_prior.surface_points.shape))
    print("empty points  :", tuple(pseudo_prior.empty_points.shape))
    print("material pts  :", tuple(pseudo_prior.material_points.shape))
    print("grid msdf idx :", tuple(pseudo_prior.material_grid_indices.shape))
    print("label stats   :")
    for k, v in sorted(pseudo_prior.label_stats.items()):
        print(f"  {k}: {v}")
    print("grid stats    :")
    for k, v in sorted(pseudo_prior.grid_stats.items()):
        print(f"  {k}: {v}")
    print("schedule:")
    for probe_it in [0, 1, 50, 100, 250, 500, 1000, 2500]:
        print(f"  iter={probe_it:4d}  xsec_w={pseudo_prior.schedule_weight(probe_it):.3f}")
else:
    print("Set FLAGS.use_pseudo_last_prior=True and provide pseudo-last paths before creating geometry.")

## 5b - Visualize one cross-section prior slice

In [ ]:
import matplotlib.pyplot as plt

pseudo_prior = getattr(geometry, "pseudo_last_prior_loss", None)
if pseudo_prior is None:
    print("No pseudo-last prior loaded.")
else:
    slice_s = globals().get("DEBUG_SLICE_S", 0.55)
    x0 = float(pseudo_prior.section_x[0].detach().cpu())
    x1 = float(pseudo_prior.section_x[-1].detach().cpu())
    x = x0 + float(slice_s) * (x1 - x0)
    tol = (x1 - x0) / max(int(FLAGS.pseudo_last_xsec_x_slices), 2) * 0.60

    def slice_points(points, max_n=5000):
        pts = points.detach().cpu().numpy()
        pts = pts[np.abs(pts[:, 0] - x) <= tol]
        if pts.shape[0] > max_n:
            choice = np.random.choice(pts.shape[0], max_n, replace=False)
            pts = pts[choice]
        return pts

    empty_pts = slice_points(pseudo_prior.empty_points)
    material_pts = slice_points(pseudo_prior.material_points)
    surface_pts = slice_points(pseudo_prior.surface_points)

    fig, ax = plt.subplots(figsize=(7, 7))
    if material_pts.size:
        ax.scatter(material_pts[:, 2], material_pts[:, 1], s=3, alpha=0.35, c="tab:orange", label="material samples")
    if empty_pts.size:
        ax.scatter(empty_pts[:, 2], empty_pts[:, 1], s=3, alpha=0.25, c="tab:blue", label="empty cavity samples")
    if surface_pts.size:
        ax.scatter(surface_pts[:, 2], surface_pts[:, 1], s=7, alpha=0.90, c="black", label="pseudo-last surface")

    ax.set_title(f"Pseudo-last cross-section samples at s={slice_s:.2f}, x={x:.4f}")
    ax.set_xlabel("z")
    ax.set_ylabel("y")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    plt.show()

## 6 - Optimizers and learning-rate schedule

In [ ]:
def resolve_learning_rates(value):
    if isinstance(value, (list, tuple)):
        if len(value) > 0 and isinstance(value[0], (list, tuple)):
            value = value[0]
        learning_rate_pos = float(value[0])
        learning_rate_mat = float(value[1]) if len(value) > 1 else learning_rate_pos
        learning_rate_lgt = float(value[2]) if len(value) > 2 else learning_rate_pos * 6.0
        return learning_rate_pos, learning_rate_mat, learning_rate_lgt
    value = float(value)
    return value, value, value * 6.0

learning_rate_pos, learning_rate_mat, learning_rate_lgt = resolve_learning_rates(FLAGS.learning_rate)
print(f"LR pos={learning_rate_pos}, mat={learning_rate_mat}, lgt={learning_rate_lgt}")

def lr_schedule(iter, fraction):
    del fraction
    return max(0.0, 10 ** (-iter * 0.0002))

image_loss_fn = createLoss(FLAGS)

params = list(material.get_parameters(mat))
optimizer = torch.optim.Adam(params, lr=learning_rate_mat)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda x: lr_schedule(x, 0.9))

optimizer_light = torch.optim.Adam((lgt.parameters() if lgt is not None else []), lr=learning_rate_lgt)
scheduler_light = torch.optim.lr_scheduler.LambdaLR(optimizer_light, lr_lambda=lambda x: lr_schedule(x, 0.9))

if FLAGS.use_sdf_mlp:
    lr_msdf = learning_rate_pos * 1e-2 if FLAGS.use_msdf_mlp else learning_rate_pos
    deform_params = [v[1] for v in geometry.named_parameters() if "deform" in v[0]]
    msdf_params = [v[1] for v in geometry.named_parameters() if "msdf" in v[0]]
    sdf_params = [v[1] for v in geometry.named_parameters() if "sdf" in v[0] and "msdf" not in v[0]]
    other_params = [v[1] for v in geometry.named_parameters() if "sdf" not in v[0] and "msdf" not in v[0] and "deform" not in v[0]]
    optimizer_mesh = torch.optim.Adam([
        {"params": deform_params, "lr": learning_rate_pos},
        {"params": msdf_params, "lr": lr_msdf},
        {"params": sdf_params, "lr": learning_rate_pos * 1e-2},
        {"params": other_params, "lr": learning_rate_pos * 1e-2},
    ], eps=1e-8)
else:
    optimizer_mesh = torch.optim.Adam(geometry.parameters(), lr=learning_rate_pos)
scheduler_mesh = torch.optim.lr_scheduler.LambdaLR(optimizer_mesh, lr_lambda=lambda x: lr_schedule(x, 0.9))

print(f"Optimizer param groups (mesh): {len(optimizer_mesh.param_groups)}")
print(f"Optimizer param groups (mat) : {len(optimizer.param_groups)}")

## 7 - Single training step debug

In [ ]:
dataloader_train = torch.utils.data.DataLoader(
    dataset_train, batch_size=FLAGS.batch, collate_fn=dataset_train.collate, shuffle=True
)
train_iter = iter(dataloader_train)

target = next(train_iter)
target = prepare_batch(target, "random")

print("Batch keys:", list(target.keys()))
print(f"  img       : {tuple(target['img'].shape)} device={target['img'].device}")
print(f"  mv        : {tuple(target['mv'].shape)}")
print(f"  mvp       : {tuple(target['mvp'].shape)}")
print(f"  campos    : {tuple(target['campos'].shape)}")
print(f"  background: {tuple(target['background'].shape)}")

In [ ]:
optimizer.zero_grad()
optimizer_mesh.zero_grad()
optimizer_light.zero_grad()

if lgt is not None:
    lgt.update_pdf()

it = int(globals().get("DEBUG_ITERATION", 0))
img_loss, depth_loss, reg_loss = geometry.tick(
    glctx, target, lgt, mat, image_loss_fn, it, denoiser=denoiser
)

total_loss = img_loss + reg_loss
xsec_stats = getattr(geometry, "last_pseudo_last_prior_stats", {})

print(f"iteration  = {it}")
print(f"img_loss   = {img_loss.item():.6f}")
print(f"depth_loss = {depth_loss.item():.6f}")
print(f"reg_loss   = {reg_loss.item():.6f}")
print(f"total_loss = {total_loss.item():.6f}")
print("cross-section pseudo-last stats:")
for key in [
    "schedule_weight",
    "xsec_loss",
    "xsec_material_loss_raw",
    "xsec_empty_loss_raw",
    "xsec_surface_loss_raw",
    "xsec_msdf_loss_raw",
    "xsec_mat_pts",
    "xsec_mat_sdf_mean",
    "xsec_mat_sdf_pos",
    "xsec_mat_sdf_margin",
    "xsec_empty_pts",
    "xsec_empty_sdf_mean",
    "xsec_empty_sdf_neg",
    "xsec_empty_sdf_margin",
    "xsec_surface_pts",
    "xsec_surface_abs_sdf",
    "xsec_grid_msdf_pts",
    "xsec_grid_msdf_pos",
]:
    if key in xsec_stats:
        print(f"  {key}: {xsec_stats[key]}")

In [ ]:
total_loss.backward()

if hasattr(lgt, "base") and lgt.base.grad is not None:
    lgt.base.grad *= 64
if "kd_ks" in mat and mat["kd_ks"].encoder.params.grad is not None:
    mat["kd_ks"].encoder.params.grad /= 8.0

print("Gradient norms after backward:")
for name, param in geometry.named_parameters():
    if param.grad is not None:
        print(f"  {name:45s} grad_norm={param.grad.norm().item():.6f}")

RUN_OPTIMIZER_STEP = bool(globals().get("RUN_OPTIMIZER_STEP", False))
if RUN_OPTIMIZER_STEP:
    optimizer.step(); scheduler.step()
    optimizer_mesh.step(); scheduler_mesh.step()
    optimizer_light.step(); scheduler_light.step()
    with torch.no_grad():
        if "kd" in mat: mat["kd"].clamp_()
        if "ks" in mat: mat["ks"].clamp_()
        if lgt is not None: lgt.clamp_(min=1e-4)
        geometry.clamp_deform()
    torch.cuda.current_stream().synchronize()
    print("Optimizer step completed.")
else:
    print("Backward done. Set RUN_OPTIMIZER_STEP=True before this cell to step parameters.")

## 8 - Short notebook training loop

In [ ]:
import tqdm

N_STEPS = int(globals().get("NOTEBOOK_TRAIN_STEPS", 100))
log_interval = int(globals().get("NOTEBOOK_LOG_INTERVAL", 10))

img_loss_vec = []
reg_loss_vec = []
xsec_loss_vec = []
xsec_mat_pos_vec = []
xsec_empty_neg_vec = []
xsec_surface_abs_vec = []
xsec_grid_msdf_pos_vec = []
iter_ms_vec = []

last_valid_state = None
last_valid_it = None

dataloader = torch.utils.data.DataLoader(
    dataset_train, batch_size=FLAGS.batch, collate_fn=dataset_train.collate, shuffle=True
)

for it, target in enumerate(tqdm.tqdm(dataloader, total=N_STEPS)):
    if it >= N_STEPS:
        break
    target = prepare_batch(target, "random")
    t0 = time.time()

    optimizer.zero_grad()
    optimizer_mesh.zero_grad()
    optimizer_light.zero_grad()

    if lgt is not None:
        lgt.update_pdf()

    try:
        img_loss, depth_loss, reg_loss = geometry.tick(
            glctx, target, lgt, mat, image_loss_fn, it, denoiser=denoiser
        )
    except render.EmptyMeshError:
        optimizer.zero_grad(); optimizer_mesh.zero_grad(); optimizer_light.zero_grad()
        if last_valid_state is not None:
            geometry.load_state_dict(last_valid_state)
            geometry.clamp_deform()
            print(f"[iter={it}] Empty mesh; restored state from iter={last_valid_it}")
        else:
            print(f"[iter={it}] Empty mesh; no valid state yet")
        continue

    total_loss = img_loss + reg_loss
    total_loss.backward()

    if hasattr(lgt, "base") and lgt.base.grad is not None:
        lgt.base.grad *= 64
    if "kd_ks" in mat and mat["kd_ks"].encoder.params.grad is not None:
        mat["kd_ks"].encoder.params.grad /= 8.0

    last_valid_state = {k: v.detach().clone() for k, v in geometry.state_dict().items()}
    last_valid_it = it

    optimizer.step(); scheduler.step()
    optimizer_mesh.step(); scheduler_mesh.step()
    optimizer_light.step(); scheduler_light.step()

    with torch.no_grad():
        if "kd" in mat: mat["kd"].clamp_()
        if "ks" in mat: mat["ks"].clamp_()
        if lgt is not None: lgt.clamp_(min=1e-4)
        geometry.clamp_deform()

    torch.cuda.current_stream().synchronize()
    iter_ms = (time.time() - t0) * 1000.0

    xsec_stats = getattr(geometry, "last_pseudo_last_prior_stats", {})
    img_loss_vec.append(float(img_loss.detach().cpu()))
    reg_loss_vec.append(float(reg_loss.detach().cpu()))
    xsec_loss_vec.append(float(xsec_stats.get("xsec_loss", 0.0)))
    xsec_mat_pos_vec.append(float(xsec_stats.get("xsec_mat_sdf_pos", np.nan)))
    xsec_empty_neg_vec.append(float(xsec_stats.get("xsec_empty_sdf_neg", np.nan)))
    xsec_surface_abs_vec.append(float(xsec_stats.get("xsec_surface_abs_sdf", np.nan)))
    xsec_grid_msdf_pos_vec.append(float(xsec_stats.get("xsec_grid_msdf_pos", np.nan)))
    iter_ms_vec.append(iter_ms)

    if it % log_interval == 0:
        tqdm.tqdm.write(
            f"iter={it:5d} img={np.mean(img_loss_vec[-log_interval:]):.6f} "
            f"reg={np.mean(reg_loss_vec[-log_interval:]):.6f} "
            f"xsec={xsec_loss_vec[-1]:.6f} mat_pos={xsec_mat_pos_vec[-1]:.3f} "
            f"empty_neg={xsec_empty_neg_vec[-1]:.3f} surf_abs={xsec_surface_abs_vec[-1]:.5f} "
            f"grid_msdf_pos={xsec_grid_msdf_pos_vec[-1]:.3f} time={iter_ms:.1f}ms"
        )

print(f"Finished {len(img_loss_vec)} notebook iterations.")

## 9 - Loss curves

In [ ]:
import matplotlib.pyplot as plt

series = [
    ("Image loss", img_loss_vec, "log"),
    ("Regularization loss", reg_loss_vec, "log"),
    ("XSec weighted loss", xsec_loss_vec, None),
    ("Material SDF positive fraction", xsec_mat_pos_vec, None),
    ("Empty SDF negative fraction", xsec_empty_neg_vec, None),
    ("Pseudo-last surface |SDF|", xsec_surface_abs_vec, None),
    ("Grid mSDF positive fraction", xsec_grid_msdf_pos_vec, None),
    ("Iteration time ms", iter_ms_vec, None),
]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.ravel()
for ax, (title, values, scale) in zip(axes, series):
    ax.plot(values, linewidth=0.9)
    ax.set_title(title)
    if scale:
        ax.set_yscale(scale)
    if "fraction" in title:
        ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 9a - Parse the running training log

In [ ]:
log_path = Path(training_log_path)
print("log exists:", log_path.exists(), log_path)

records = []
if log_path.exists():
    pattern = re.compile(
        r"iter=\s*(?P<iter>\d+).*?img_loss=(?P<img>[0-9.eE+-]+).*?reg_loss=(?P<reg>[0-9.eE+-]+).*?"
        r"xsec_loss=(?P<xsec>[0-9.eE+-]+).*?xsec_w=(?P<w>[0-9.eE+-]+).*?"
        r"xsec_mat_pts=(?P<mat_pts>\d+).*?xsec_mat_pos=(?P<mat_pos>[0-9.eE+-]+).*?"
        r"xsec_empty_pts=(?P<empty_pts>\d+).*?xsec_empty_neg=(?P<empty_neg>[0-9.eE+-]+).*?"
        r"xsec_surface_pts=(?P<surface_pts>\d+).*?xsec_surface_abs=(?P<surface_abs>[0-9.eE+-]+).*?"
        r"xsec_grid_msdf_pts=(?P<grid_pts>\d+).*?xsec_grid_msdf_pos=(?P<grid_pos>[0-9.eE+-]+)"
    )
    for line in log_path.read_text(errors="replace").splitlines():
        m = pattern.search(line)
        if m:
            row = {k: float(v) for k, v in m.groupdict().items()}
            row["iter"] = int(row["iter"])
            row["mat_pts"] = int(row["mat_pts"])
            row["empty_pts"] = int(row["empty_pts"])
            row["surface_pts"] = int(row["surface_pts"])
            row["grid_pts"] = int(row["grid_pts"])
            records.append(row)

print("parsed rows:", len(records))
if records:
    print("last row:")
    for k, v in records[-1].items():
        print(f"  {k}: {v}")

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    xs = [r["iter"] for r in records]
    axes[0].plot(xs, [r["img"] for r in records], label="img")
    axes[0].plot(xs, [r["reg"] for r in records], label="reg")
    axes[0].set_yscale("log")
    axes[0].legend(); axes[0].grid(True, alpha=0.25); axes[0].set_title("losses")
    axes[1].plot(xs, [r["xsec"] for r in records]); axes[1].grid(True, alpha=0.25); axes[1].set_title("xsec loss")
    axes[2].plot(xs, [r["mat_pos"] for r in records], label="material positive")
    axes[2].plot(xs, [r["empty_neg"] for r in records], label="empty negative")
    axes[2].plot(xs, [r["grid_pos"] for r in records], label="grid mSDF positive")
    axes[2].set_ylim(-0.05, 1.05); axes[2].legend(); axes[2].grid(True, alpha=0.25); axes[2].set_title("fractions")
    axes[3].plot(xs, [r["surface_abs"] for r in records]); axes[3].grid(True, alpha=0.25); axes[3].set_title("surface |SDF|")
    plt.tight_layout()
    plt.show()

## 10 - Extract and inspect mesh

In [ ]:
with torch.no_grad():
    result = geometry.getMesh(mat)
    base_mesh = result["imesh"]

    print(f"Open vertices       : {tuple(base_mesh.v_pos.shape)}")
    print(f"Open triangles      : {tuple(base_mesh.t_pos_idx.shape)}")
    print(f"Watertight vertices : {tuple(result['vertices_watertight'].shape)}")
    print(f"Watertight faces    : {tuple(result['faces_watertight'].shape)}")
    print(f"mSDF watertight     : {tuple(result['msdf_watertight'].shape)}")
    if base_mesh.v_pos.numel() > 0:
        print("v_pos min:", base_mesh.v_pos.min(dim=0).values.detach().cpu().numpy())
        print("v_pos max:", base_mesh.v_pos.max(dim=0).values.detach().cpu().numpy())
    print("grid_msdf raw       :", tuple(result["grid_msdf_raw"].shape))
    print("grid_sdf            :", tuple(result["grid_sdf"].shape))

## 10a - Probe current shell SDF on pseudo-last samples

In [ ]:
pseudo_prior = getattr(geometry, "pseudo_last_prior_loss", None)
if pseudo_prior is None:
    print("No pseudo-last prior loaded.")
else:
    with torch.no_grad():
        def shell_sdf(points_world):
            query_points = points_world
            if torch.is_tensor(geometry.offset):
                query_points = query_points - geometry.offset.to(device=query_points.device, dtype=query_points.dtype)
            return geometry.sdf_net(query_points).reshape(-1)

        mat_points = pseudo_prior._sample_rows(pseudo_prior.material_points, 20000)
        empty_points = pseudo_prior._sample_rows(pseudo_prior.empty_points, 20000)
        surface_points = pseudo_prior._sample_rows(pseudo_prior.surface_points, 8192)
        mat_sdf = shell_sdf(mat_points)
        empty_sdf = shell_sdf(empty_points)
        surface_sdf = shell_sdf(surface_points)

        print("material samples:", tuple(mat_points.shape))
        print("  mean SDF      :", float(mat_sdf.mean().cpu()))
        print("  positive frac :", float((mat_sdf > 0).float().mean().cpu()))
        print("  margin frac   :", float((mat_sdf >= FLAGS.pseudo_last_xsec_material_margin).float().mean().cpu()))
        print("empty samples   :", tuple(empty_points.shape))
        print("  mean SDF      :", float(empty_sdf.mean().cpu()))
        print("  negative frac :", float((empty_sdf < 0).float().mean().cpu()))
        print("  margin frac   :", float((empty_sdf <= -FLAGS.pseudo_last_xsec_empty_margin).float().mean().cpu()))
        print("surface samples :", tuple(surface_points.shape))
        print("  mean |SDF|    :", float(surface_sdf.abs().mean().cpu()))

        if pseudo_prior.material_grid_indices.numel() > 0:
            grid_msdf = result["grid_msdf_raw"].reshape(-1)
            selected = grid_msdf[pseudo_prior.material_grid_indices.to(grid_msdf.device)]
            print("grid mSDF samples:", int(selected.numel()))
            print("  mean          :", float(selected.mean().detach().cpu()))
            print("  positive frac :", float((selected > 0).float().mean().detach().cpu()))

## 11 - Render a validation view and compare to GT

In [ ]:
dataloader_validate = torch.utils.data.DataLoader(
    dataset_validate, batch_size=1, collate_fn=dataset_validate.collate
)

val_target = next(iter(dataloader_validate))
val_target = prepare_batch(val_target, FLAGS.background)

result_image, result_dict = validate_itr(glctx, val_target, geometry, mat, lgt, FLAGS, denoiser=denoiser)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
opt_img = result_dict["opt"].detach().cpu().numpy()
ref_img = result_dict["ref"].detach().cpu().numpy()
axes[0].imshow(np.clip(opt_img, 0, 1)); axes[0].set_title("Rendered"); axes[0].axis("off")
axes[1].imshow(np.clip(ref_img, 0, 1)); axes[1].set_title("Ground Truth"); axes[1].axis("off")
plt.tight_layout()
plt.show()

## 12 - Save debug outputs

In [ ]:
with torch.no_grad():
    mesh_dir = os.path.join(output_dir, "mesh")
    os.makedirs(mesh_dir, exist_ok=True)

    torch.save(geometry.state_dict(), os.path.join(mesh_dir, "model.pt"))
    torch.save(mat["kd_ks"].state_dict(), os.path.join(mesh_dir, "mtl.pt"))
    if lgt is not None:
        light.save_env_map(os.path.join(mesh_dir, "probe.hdr"), lgt)

    export_mesh = geometry.getMesh(mat)["imesh"]
    obj.write_obj(os.path.join(mesh_dir, ""), export_mesh, save_material=False)

    print(f"Saved to {mesh_dir}/")
    for f in sorted(os.listdir(mesh_dir)):
        fpath = os.path.join(mesh_dir, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f:30s} {size_mb:.2f} MB")

## 13 - Scratch cell

In [ ]:
# Example: inspect gradient norms per parameter.
for name, param in geometry.named_parameters():
    grad_norm = param.grad.norm().item() if param.grad is not None else 0.0
    print(f"  {name:45s} val_norm={param.data.norm().item():.4f} grad_norm={grad_norm:.6f}")